# 02 - Concurrency probe (GPU, internet OFF)

Accelerator: **`machine_shape: NvidiaL4`** (pinned in `kernel-metadata.json`,
so it survives every CLI push). Budget: **~1 GPU-hour**.

## Why internet is off

Pushing with `NvidiaL4` + `enable_internet: true` is rejected:

> `Internet cannot be enabled for this competition with the current accelerator.`

That rules out the PyPI dependency staging the calibration runs rely on. Which
is fine, because **Phase 1 does not need the repos' test dependencies** — only
Phase 2 verification does, and verification needs no GPU at all (gold
calibration measured it at p50 8.1s/task on CPU).

So the pipeline splits along the resource it actually needs:

| Stage | Where | Internet | Needs GPU |
|---|---|---|---|
| Phase 1 — agent writes a patch | this GPU kernel | off | **yes** |
| Phase 2 — verify the patch | CPU kernel (as in calibration) | on | no |

This probe therefore measures **apparatus and throughput**, judged on
*patch produced*, not *task resolved*. Resolution comes from a later CPU pass
over the saved patches.

Both stages cap the per-task budget hard (10 min, 30 tool calls); without a cap
one task can occupy 180 minutes. These numbers are **not comparable to a real
measurement run**.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free',
                      '--format=csv'], capture_output=True, text=True).stdout)
print('CPUs:', os.cpu_count())
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
print(subprocess.run(['df', '-h', '/kaggle/working', '/tmp'], capture_output=True, text=True).stdout)

**Check the output above shows 4 x L4.**

`enable_gpu: true` alone gave us **2 x Tesla T4** on the first run — the
boolean cannot express a GPU type. The accelerator is now pinned via
`machine_shape: NvidiaL4` in `kernel-metadata.json`, which the CLI reads on
every push, so it no longer depends on a browser setting.

T4 could never have worked: W4A16 compressed-tensors needs Ampere (8.0+) and
T4 is Turing (7.5), quite apart from 30 GB being too little for 31B weights
plus KV cache at 32k context.

In [ ]:
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

INPUT = Path('/kaggle/input')
print('attached:', [p.name for p in sorted(INPUT.iterdir())])


def find_dir(*needles, must_contain=''):
    c = [p for p in INPUT.rglob('*')
         if p.is_dir() and any(n in p.name.lower() for n in needles)
         and (not must_contain or any(p.glob(must_contain)))]
    return sorted(c, key=lambda q: len(str(q)))[0] if c else None


def find_model_dir(slug):
    """Return the directory that actually holds the weights.

    Kaggle mounts a model at <owner>/<model>/<framework>/<instance>/<version>,
    so matching on the instance name alone lands one level too high and vLLM
    rejects it with 'Invalid repository ID or local directory'. Anchor on
    config.json instead of guessing the version number.
    """
    hits = [q.parent for q in INPUT.rglob('config.json') if slug in str(q.parent).lower()]
    if not hits:
        hits = [q.parent for q in INPUT.rglob('config.json')
                if any(q.parent.glob('*.safetensors'))]
    return sorted(hits, key=lambda q: len(str(q)))[0] if hits else None


WHEELHOUSE = find_dir('wheelhouse', must_contain='*.whl')
LAB = find_dir('swegemma-lab')
DATA = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL = find_model_dir('gemma-4-31b-it-qat-w4a16-ct')
print('wheelhouse:', WHEELHOUSE)
print('lab       :', LAB)
print('data      :', DATA, DATA.is_dir())
print('model     :', MODEL)
assert WHEELHOUSE and LAB and DATA.is_dir() and MODEL, 'missing an attached source'
assert (MODEL / 'config.json').is_file(), f'no config.json in {MODEL}'
print('model files:', sorted(q.name for q in MODEL.iterdir())[:8])

In [ ]:
# Remove broken cutlass .pth hooks that break imports on this image.
import glob
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
            '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for f in glob.glob(pat):
        try:
            os.unlink(f)
        except OSError:
            pass

tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    n = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp / n).exists():
        os.symlink(w, tmp / n)

wheels = sorted(str(w) for w in tmp.glob('*.whl'))
print(f'installing {len(wheels)} wheels')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    '--force-reinstall', *wheels], capture_output=True, text=True)
print('pip rc', r.returncode, (r.stderr or '')[-800:])

In [ ]:
# STAGE A -- does any of this work? 4 tasks, concurrency 2.
#
# subprocess.run(timeout=) is the only brake that exists: Kaggle has no
# 'cancel kernel' in the CLI, and run 1 of this probe idled for the full 12-hour
# session cap after vLLM failed to start -- a quarter of the weekly quota spent
# on nothing. Bound every stage, and stop the notebook if a stage fails.
RUNNER = str(LAB / 'scripts' / 'session_runner.py')
COMMON = ['--lab', str(LAB), '--data', str(DATA), '--model-path', str(MODEL),
          '--variant', 'v00_public_baseline', '--task-list', 'clean_n.txt',
          '--max-time-minutes', '10', '--max-tool-calls', '30']

STAGE_TIMEOUT_S = 50 * 60   # vLLM startup (~6 min observed) + 4 tasks at <=10 min


def run_stage(run_id, limit, conc, timeout_s=STAGE_TIMEOUT_S):
    t0 = time.time()
    try:
        rc = subprocess.run(
            [sys.executable, RUNNER, *COMMON, '--run-id', run_id,
             '--limit', str(limit), '--concurrency', str(conc)],
            timeout=timeout_s,
        ).returncode
    except subprocess.TimeoutExpired:
        rc = -9
        print(f'TIMEOUT: {run_id} exceeded {timeout_s/60:.0f} min and was killed')
    wall = time.time() - t0
    print(f'\n{run_id}: rc={rc}  wall={wall/60:.1f} min')
    return rc, wall

rcA, aw = run_stage('probe_c2', 4, 2)
assert rcA == 0, (
    'Stage A failed -- stopping so the session cannot idle out the quota. '
    'Read the traceback above before re-running.'
)

**The assert above stops the notebook if Stage A failed.** That is deliberate:
run 1 failed at vLLM startup and then held the GPU for the full 12-hour session
cap, spending ~12 of 45 weekly GPU-hours on nothing. There is no
`kaggle kernels cancel`, so an in-notebook timeout plus a hard assert is the
only brake available.

Failures seen so far, in order of how they presented:

| Symptom | Cause |
|---|---|
| `Invalid repository ID or local directory` | model path pointed at the instance dir, not its version subdir |
| 2 x Tesla T4 instead of L4 | `enable_gpu` is a boolean; needs `machine_shape: NvidiaL4` |
| `Internet cannot be enabled ... with the current accelerator` | L4 forbids internet on this competition |
| `CUDA OOM warming up sampler with 256 dummy requests` | default `max_num_seqs`; the engine only supports ~10x concurrency at 32k |

In [ ]:
# STAGE B -- throughput. 12 tasks, concurrency 12.
# Only reached if Stage A passed.
rcB, bw = run_stage('probe_c12', 12, 12, timeout_s=40 * 60)
print('stage B rc', rcB)

In [ ]:
# Judge on patch-produced, not resolved: Phase 2 cannot verify here, because
# the repos' test dependencies need internet, which L4 forbids.
import json
from pathlib import Path

for run_id, n, conc, wall in (('probe_c2', 4, 2, aw), ('probe_c12', 12, 12, bw)):
    base = Path('/kaggle/working/results') / run_id
    jl = base / 'task_results.jsonl'
    rows = []
    if jl.is_file():
        for line in jl.read_text(encoding='utf-8').splitlines():
            if line.strip():
                rows.append(json.loads(line))
    pdir = base / 'patches'
    patches = sorted(pdir.glob('*.patch')) if pdir.is_dir() else []
    non_empty = [q for q in patches if q.stat().st_size > 0]
    print('--- ' + run_id + ': concurrency ' + str(conc) + ' ---')
    print('    tasks done   ' + str(len(rows)) + '/' + str(n))
    print('    wall         ' + format(wall / 60, '.1f') + ' min')
    print('    patches      ' + str(len(non_empty)) + ' non-empty of ' + str(len(patches)))
    if rows:
        print('    per task     ' + format(wall / len(rows) / 60, '.2f') + ' min')
        print('    tasks/GPU-h  ' + format(len(rows) / (wall / 3600), '.1f'))
    print('')

if aw and bw:
    print('per-task speedup c2 -> c12: ' + format((aw / 4) / (bw / 12), '.2f') + 'x')
    print('extrapolated clean-N (102 tasks) at c12: '
          + format(102 * (bw / 12) / 3600, '.1f') + ' GPU-h')

In [ ]:
import shutil
for d in ('ledger', 'results'):
    p = Path('/kaggle/working') / d
    if p.is_dir():
        shutil.make_archive(f'/kaggle/working/probe_{d}', 'zip', root_dir=p)
        print('wrote', f'/kaggle/working/probe_{d}.zip')
print('\npull locally with:')
print('  kaggle kernels output ahmedsherif382/swegemma-concurrency-probe -p results/probe')